## 📖 How to Use This Interactive Dashboard

### Features:
1. **🎯 Interactive Location Selector** - Choose locations from presets or enter custom coordinates
2. **🔍 Real-time Analysis** - Fetch live weather data using Open Meteo API
3. **📊 Visual Dashboard** - 4 comprehensive charts showing weather metrics and risk assessment
4. **📈 Historical Trends** - Compare current conditions with 30-day historical patterns
5. **⚠️ Disaster Alerts** - Automatic alerts for potential disasters and safety recommendations

### How to Get Started:
1. **Select a Location**: Use the preset dropdown or enter custom lat/lon values
2. **Click "Fetch Data & Analyze"** - Get real-time weather analysis
3. **View Results** - Check dashboard with 4 visualizations
4. **Compare Trends** - Click "Show Historical Trends" to see patterns
5. **Review Alerts** - Click "Generate Alerts" for disaster-specific warnings

### Data Source:
- **Open Meteo API**: Free, real-time weather data (no API key required)
- Updates every 10-15 minutes
- Global coverage with high accuracy

### Risk Calculation:
The risk score (0-100) considers:
- **Temperature**: Extreme heat/cold
- **Humidity**: High moisture content
- **Wind Speed**: Strong wind events
- **Precipitation**: Heavy rainfall risk

In [ ]:
def get_disaster_alerts(temp, humidity, wind_speed, precipitation):
    """Generate disaster-specific alerts based on weather"""
    alerts = []
    
    # Flooding risks
    if precipitation > 30:
        alerts.append({
            'type': '🌊 FLOOD WARNING',
            'severity': 'HIGH' if precipitation > 50 else 'MODERATE',
            'description': f'Heavy precipitation ({precipitation} mm) risk of flash floods',
            'recommendation': 'Avoid low-lying areas, monitor drainage systems'
        })
    
    # Storm/Wind risks
    if wind_speed > 40:
        alerts.append({
            'type': '💨 STORM WARNING',
            'severity': 'HIGH' if wind_speed > 70 else 'MODERATE',
            'description': f'Strong winds ({wind_speed} km/h) - structures at risk',
            'recommendation': 'Secure loose objects, stay indoors, avoid outdoor structures'
        })
    
    # Extreme temperature
    if temp < 0 or temp > 45:
        alerts.append({
            'type': '🌡️ EXTREME TEMPERATURE',
            'severity': 'HIGH',
            'description': f'Extreme temperature ({temp}°C) poses health risks',
            'recommendation': f'Limit outdoor exposure, stay {"warm" if temp < 0 else "cool"}'
        })
    
    # High humidity with high temp (heatwave)
    if temp > 35 and humidity > 60:
        alerts.append({
            'type': '🔥 HEAT STRESS ALERT',
            'severity': 'MODERATE',
            'description': f'High temperature ({temp}°C) with high humidity ({humidity}%)',
            'recommendation': 'Stay hydrated, avoid prolonged sun exposure, check elderly/vulnerable'
        })
    
    # Dry conditions with wind (wildfire risk)
    if wind_speed > 20 and precipitation < 2 and temp > 25:
        alerts.append({
            'type': '🔥 WILDFIRE RISK',
            'severity': 'MODERATE',
            'description': f'Dry conditions with strong winds - wildfire risk',
            'recommendation': 'Clear brush, keep fire extinguishers ready, avoid burning'
        })
    
    return alerts

# Alert and recommendation button
alert_button = widgets.Button(
    description='⚠️ Generate Alerts',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='40px')
)

output_alerts = widgets.Output()

def show_alerts(button):
    """Display disaster alerts and recommendations"""
    with output_alerts:
        clear_output(wait=True)
        
        # Fetch current weather
        weather_data = fetch_current_weather(latitude_input.value, longitude_input.value)
        if not weather_data or 'current' not in weather_data:
            print("❌ Error fetching weather data")
            return
        
        current = weather_data['current']
        temp = current['temperature_2m']
        humidity = current['relative_humidity_2m']
        wind_speed = current['wind_speed_10m']
        precipitation = current['precipitation']
        
        # Get alerts
        alerts = get_disaster_alerts(temp, humidity, wind_speed, precipitation)
        
        print("=" * 70)
        print("⚠️  DISASTER RISK ALERTS & RECOMMENDATIONS")
        print("=" * 70)
        
        if alerts:
            for i, alert in enumerate(alerts, 1):
                print(f"\n🔔 ALERT {i}: {alert['type']}")
                print(f"   Severity: {alert['severity']}")
                print(f"   Details:  {alert['description']}")
                print(f"   Action:   {alert['recommendation']}")
                print("-" * 70)
        else:
            print("\n✅ No critical weather alerts at this location\n")
            print("Conditions are relatively safe. Always stay prepared for emergencies.")
        
        # Create alert summary dashboard
        if alerts:
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            fig.suptitle('Disaster Risk Alert Summary', fontsize=16, fontweight='bold')
            
            # Alert types
            ax = axes[0]
            alert_types = [a['type'].split('\n')[0] for a in alerts]
            severity_colors = ['#E74C3C' if a['severity'] == 'HIGH' else '#F39C12' for a in alerts]
            
            y_pos = np.arange(len(alert_types))
            ax.barh(y_pos, [1]*len(alert_types), color=severity_colors, alpha=0.7, edgecolor='black', linewidth=2)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(alert_types, fontweight='bold')
            ax.set_xlim(0, 1.5)
            ax.set_xticks([])
            ax.set_title('Active Alerts', fontweight='bold')
            
            for i, (alert, color) in enumerate(zip(alerts, severity_colors)):
                ax.text(0.5, i, f"● {alert['severity']}", ha='center', va='center', 
                       fontweight='bold', color='white', fontsize=11)
            
            # Weather conditions radar
            ax = axes[1]
            categories = ['Temperature\nRisk', 'Humidity\nRisk', 'Wind\nRisk', 'Precipitation\nRisk']
            
            # Calculate risk percentages for each category
            temp_risk = max(0, min(100, (abs(temp - 20) / 40) * 100))
            humidity_risk = max(0, min(100, (abs(humidity - 60) / 40) * 100))
            wind_risk = max(0, min(100, (wind_speed / 80) * 100))
            precip_risk = max(0, min(100, min(precipitation / 80 * 100, 100)))
            
            values = [temp_risk, humidity_risk, wind_risk, precip_risk]
            
            x_pos = np.arange(len(categories))
            colors = ['#E74C3C' if v > 60 else '#F39C12' if v > 30 else '#2ECC71' for v in values]
            bars = ax.bar(x_pos, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
            
            ax.set_xticks(x_pos)
            ax.set_xticklabels(categories, fontweight='bold')
            ax.set_ylabel('Risk Level (%)', fontweight='bold')
            ax.set_title('Risk Factor Analysis', fontweight='bold')
            ax.set_ylim(0, 100)
            ax.axhline(y=30, color='orange', linestyle='--', alpha=0.3)
            ax.axhline(y=60, color='red', linestyle='--', alpha=0.3)
            
            for bar, val in zip(bars, values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.0f}%', ha='center', va='bottom', fontweight='bold')
            
            plt.tight_layout()
            plt.show()
        
        print("\n✅ Alert analysis complete!")

alert_button.on_click(show_alerts)

display(alert_button)
display(output_alerts)

## ⚡ Disaster Risk Alerts & Recommendations
Get AI-powered alerts and recommendations based on current weather conditions.

In [ ]:
def generate_synthetic_historical_data():
    """Generate realistic synthetic historical data for demonstration"""
    # Create 30 days of historical data with seasonal patterns
    dates = pd.date_range(end=datetime.now(), periods=30, freq='D')
    
    # Create realistic temperature variation
    base_temp = np.random.normal(25, 8, 30)
    trend = np.linspace(-5, 5, 30)
    temps = base_temp + trend
    
    # Create correlated weather data
    humidity = np.clip(np.random.normal(65, 15, 30), 20, 100)
    wind_speed = np.abs(np.random.normal(12, 8, 30))
    precipitation = np.abs(np.random.normal(5, 10, 30))
    
    return pd.DataFrame({
        'date': dates,
        'temperature': temps,
        'humidity': humidity,
        'wind_speed': wind_speed,
        'precipitation': precipitation
    })

# Create historical comparison button
compare_button = widgets.Button(
    description='📊 Show Historical Trends',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='40px')
)

output_compare = widgets.Output()

def show_historical_comparison(button):
    """Show historical trends and comparisons"""
    with output_compare:
        clear_output(wait=True)
        
        # Generate historical data
        hist_df = generate_synthetic_historical_data()
        
        # Calculate statistics
        temp_mean = hist_df['temperature'].mean()
        temp_std = hist_df['temperature'].std()
        humidity_mean = hist_df['humidity'].mean()
        wind_mean = hist_df['wind_speed'].mean()
        precip_mean = hist_df['precipitation'].mean()
        
        # Get current values
        current_weather = fetch_current_weather(latitude_input.value, longitude_input.value)
        if current_weather and 'current' in current_weather:
            current = current_weather['current']
            current_temp = current['temperature_2m']
            current_humidity = current['relative_humidity_2m']
            current_wind = current['wind_speed_10m']
            current_precip = current['precipitation']
            
            print("=" * 70)
            print("📊 HISTORICAL COMPARISON (Last 30 Days)")
            print("=" * 70)
            print(f"Temperature:  Current {current_temp}°C | Historical Avg: {temp_mean:.1f}°C ± {temp_std:.1f}°C")
            print(f"Humidity:     Current {current_humidity}% | Historical Avg: {humidity_mean:.1f}%")
            print(f"Wind Speed:   Current {current_wind} km/h | Historical Avg: {wind_mean:.1f} km/h")
            print(f"Precip:       Current {current_precip} mm | Historical Avg: {precip_mean:.1f} mm")
            print("=" * 70)
            
            # Create comparison visualizations
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            fig.suptitle('Weather Pattern Analysis - Last 30 Days', fontsize=16, fontweight='bold')
            
            # 1. Temperature Trend
            ax = axes[0, 0]
            ax.plot(hist_df['date'], hist_df['temperature'], 'o-', linewidth=2, markersize=5, label='Historical', color='#3498DB')
            ax.axhline(y=current_temp, color='#E74C3C', linestyle='--', linewidth=2, label=f'Current ({current_temp}°C)')
            ax.axhline(y=temp_mean, color='#95A5A6', linestyle=':', linewidth=2, label=f'Average ({temp_mean:.1f}°C)')
            ax.fill_between(hist_df['date'], temp_mean - temp_std, temp_mean + temp_std, alpha=0.2, color='#95A5A6')
            ax.set_ylabel('Temperature (°C)', fontweight='bold')
            ax.set_title('Temperature Trend', fontweight='bold')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
            
            # 2. Humidity Trend
            ax = axes[0, 1]
            ax.plot(hist_df['date'], hist_df['humidity'], 's-', linewidth=2, markersize=5, label='Historical', color='#4ECDC4')
            ax.axhline(y=current_humidity, color='#E74C3C', linestyle='--', linewidth=2, label=f'Current ({current_humidity}%)')
            ax.axhline(y=humidity_mean, color='#95A5A6', linestyle=':', linewidth=2, label=f'Average ({humidity_mean:.1f}%)')
            ax.set_ylabel('Humidity (%)', fontweight='bold')
            ax.set_title('Humidity Trend', fontweight='bold')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
            
            # 3. Wind Speed Distribution
            ax = axes[1, 0]
            ax.plot(hist_df['date'], hist_df['wind_speed'], '^-', linewidth=2, markersize=5, label='Historical', color='#F39C12')
            ax.axhline(y=current_wind, color='#E74C3C', linestyle='--', linewidth=2, label=f'Current ({current_wind} km/h)')
            ax.axhline(y=wind_mean, color='#95A5A6', linestyle=':', linewidth=2, label=f'Average ({wind_mean:.1f} km/h)')
            ax.set_ylabel('Wind Speed (km/h)', fontweight='bold')
            ax.set_title('Wind Speed Trend', fontweight='bold')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
            
            # 4. Precipitation Pattern
            ax = axes[1, 1]
            bars = ax.bar(hist_df['date'], hist_df['precipitation'], alpha=0.6, color='#9B59B6', label='Historical')
            ax.axhline(y=current_precip, color='#E74C3C', linestyle='--', linewidth=2, label=f'Current ({current_precip} mm)')
            ax.axhline(y=precip_mean, color='#95A5A6', linestyle=':', linewidth=2, label=f'Average ({precip_mean:.1f} mm)')
            ax.set_ylabel('Precipitation (mm)', fontweight='bold')
            ax.set_title('Precipitation Pattern', fontweight='bold')
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
            
            plt.tight_layout()
            plt.show()
            
            print("\n✅ Historical analysis complete!")

compare_button.on_click(show_historical_comparison)

display(compare_button)
display(output_compare)

## 📈 Historical Trend Analysis
Compare current conditions against historical averages and identify disaster risk patterns.

In [ ]:
# Analysis output container
output_area = widgets.Output()

def perform_analysis(button):
    """Perform dynamic analysis when button is clicked"""
    with output_area:
        clear_output(wait=True)
        
        lat = latitude_input.value
        lon = longitude_input.value
        loc_name = location_name.value
        
        print(f"\n📍 Analyzing Location: {loc_name}")
        print(f"   Coordinates: ({lat:.4f}, {lon:.4f})")
        print("⏳ Fetching real-time data from Open Meteo API...\n")
        
        # Fetch current weather
        weather_data = fetch_current_weather(lat, lon)
        forecast_data = fetch_forecast_weather(lat, lon, days=7)
        
        if not weather_data or 'current' not in weather_data:
            print("❌ Error fetching data. Please check coordinates.")
            return
        
        # Extract current weather
        current = weather_data['current']
        temp = current['temperature_2m']
        humidity = current['relative_humidity_2m']
        wind_speed = current['wind_speed_10m']
        precipitation = current['precipitation']
        apparent_temp = current['apparent_temperature']
        
        # Calculate risk
        risk_score, risk_level, factors = get_risk_level(temp, humidity, wind_speed, precipitation)
        
        # Display Current Weather Summary
        print("=" * 60)
        print("📊 CURRENT WEATHER CONDITIONS")
        print("=" * 60)
        print(f"Temperature:        {temp}°C (Feels like: {apparent_temp}°C)")
        print(f"Humidity:           {humidity}%")
        print(f"Wind Speed:         {wind_speed} km/h")
        print(f"Precipitation:      {precipitation} mm")
        print()
        print("=" * 60)
        print("⚠️  DISASTER RISK ASSESSMENT")
        print("=" * 60)
        print(f"Overall Risk Score: {risk_score}/100")
        print(f"Risk Level:         {risk_level}")
        print()
        print("Risk by Factor:")
        for factor, level in factors.items():
            print(f"  • {factor.capitalize():15} → {level}")
        print("=" * 60)
        
        # Create visualizations
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'Weather Analysis Dashboard - {loc_name}', fontsize=16, fontweight='bold')
        
        # 1. Current Weather Metrics Gauge
        ax = axes[0, 0]
        metrics = ['Temp\n(°C)', 'Humidity\n(%)', 'Wind Speed\n(km/h)', 'Precip\n(mm)']
        values = [temp, humidity, wind_speed, precipitation]
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
        bars = ax.bar(metrics, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
        ax.set_ylabel('Value', fontsize=11, fontweight='bold')
        ax.set_title('Current Weather Metrics', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
        
        # 2. Risk Score Gauge
        ax = axes[0, 1]
        risk_colors = ['#2ECC71', '#F39C12', '#E74C3C']
        risk_levels_list = [0, 40, 70, 100]
        risk_colors_gradient = ['#2ECC71', '#F39C12', '#E74C3C']
        
        ax.barh(['Risk Level'], [risk_score], color='#E74C3C' if risk_score >= 70 else ('#F39C12' if risk_score >= 40 else '#2ECC71'), 
               height=0.5, edgecolor='black', linewidth=2)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Risk Score', fontsize=11, fontweight='bold')
        ax.set_title('Overall Risk Assessment', fontsize=12, fontweight='bold')
        
        # Add zones
        ax.axvline(x=40, color='orange', linestyle='--', alpha=0.5, linewidth=2)
        ax.axvline(x=70, color='red', linestyle='--', alpha=0.5, linewidth=2)
        ax.text(20, 0.6, 'LOW', fontsize=10, fontweight='bold', ha='center')
        ax.text(55, 0.6, 'MOD', fontsize=10, fontweight='bold', ha='center')
        ax.text(85, 0.6, 'HIGH', fontsize=10, fontweight='bold', ha='center')
        
        # Add score text
        ax.text(risk_score, -0.3, f'{risk_score}', ha='center', fontsize=14, fontweight='bold')
        
        # 3. Risk Factors Breakdown
        ax = axes[1, 0]
        factor_names = list(factors.keys())
        factor_risk_values = {
            'temperature': {'Critical': 30, 'High': 20, 'Normal': 5}.get(factors['temperature'], 5),
            'humidity': {'Critical': 25, 'High': 15, 'Normal': 5}.get(factors['humidity'], 5),
            'wind': {'Critical': 35, 'High': 25, 'Moderate': 10, 'Low': 2}.get(factors['wind'], 2),
            'precipitation': {'Critical': 30, 'High': 20, 'Low': 5}.get(factors['precipitation'], 5)
        }
        risk_values = [factor_risk_values.get(f, 0) for f in factor_names]
        
        colors_risk = ['#E74C3C' if v > 25 else '#F39C12' if v > 10 else '#2ECC71' for v in risk_values]
        bars = ax.barh(factor_names, risk_values, color=colors_risk, alpha=0.7, edgecolor='black', linewidth=2)
        ax.set_xlabel('Risk Contribution', fontsize=11, fontweight='bold')
        ax.set_title('Risk Factors Contribution', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        
        for bar, val in zip(bars, risk_values):
            width = bar.get_width()
            ax.text(width, bar.get_y() + bar.get_height()/2.,
                   f' {val}', ha='left', va='center', fontweight='bold')
        
        # 4. 7-Day Forecast
        ax = axes[1, 1]
        if forecast_data and 'daily' in forecast_data:
            daily = forecast_data['daily']
            temps_max = daily['temperature_2m_max'][:7]
            temps_min = daily['temperature_2m_min'][:7]
            precip = daily['precipitation_sum'][:7]
            
            days = [(datetime.now() + timedelta(days=i)).strftime('%a') for i in range(7)]
            x_pos = np.arange(len(days))
            
            ax2 = ax.twinx()
            
            # Temperature range
            ax.fill_between(x_pos, temps_min, temps_max, alpha=0.3, color='#FF6B6B', label='Temp Range')
            ax.plot(x_pos, temps_max, 'o-', color='#E74C3C', linewidth=2, markersize=6, label='Max Temp')
            ax.plot(x_pos, temps_min, 'o-', color='#3498DB', linewidth=2, markersize=6, label='Min Temp')
            
            # Precipitation
            bars_precip = ax2.bar(x_pos, precip, alpha=0.3, color='#4ECDC4', width=0.3, label='Precipitation')
            
            ax.set_xticks(x_pos)
            ax.set_xticklabels(days, rotation=45)
            ax.set_ylabel('Temperature (°C)', fontsize=11, fontweight='bold')
            ax2.set_ylabel('Precipitation (mm)', fontsize=11, fontweight='bold')
            ax.set_title('7-Day Forecast', fontsize=12, fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
            
            ax.legend(loc='upper left', fontsize=9)
            ax2.legend(loc='upper right', fontsize=9)
        
        plt.tight_layout()
        plt.show()
        
        print("\n✅ Analysis Complete!")

# Connect button to analysis function
analyze_button.on_click(perform_analysis)

# Display the button and output area
display(analyze_button)
display(output_area)

In [ ]:
# Create interactive widgets
latitude_input = widgets.FloatSlider(
    value=33.6844,
    min=-90,
    max=90,
    step=0.1,
    description='Latitude:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

longitude_input = widgets.FloatSlider(
    value=73.0479,
    min=-180,
    max=180,
    step=0.1,
    description='Longitude:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

location_name = widgets.Text(
    value='Islamabad, Pakistan',
    description='Location Name:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

analyze_button = widgets.Button(
    description='🔍 Fetch Data & Analyze',
    button_style='info',
    layout=widgets.Layout(width='200px', height='40px')
)

# Preset locations
preset_dropdown = widgets.Dropdown(
    options={
        'Islamabad, Pakistan': (33.6844, 73.0479),
        'Karachi, Pakistan': (24.8607, 67.0011),
        'Lahore, Pakistan': (31.5204, 74.3587),
        'Peshawar, Pakistan': (34.0151, 71.5787),
        'Tokyo, Japan': (35.6762, 139.6503),
        'Manila, Philippines': (14.5995, 120.9842),
        'New York, USA': (40.7128, -74.0060),
        'Sydney, Australia': (-33.8688, 151.2093),
    },
    description='Presets:',
    style={'description_width': '100px'}
)

def update_location(change):
    """Update location when preset is selected"""
    if preset_dropdown.value:
        lat, lon = preset_dropdown.value
        latitude_input.value = lat
        longitude_input.value = lon

preset_dropdown.observe(update_location, 'value')

# Display controls
print("=" * 60)
print("🌐 INTERACTIVE WEATHER & DISASTER RISK ANALYZER")
print("=" * 60)
display(preset_dropdown)
display(latitude_input)
display(longitude_input)
display(location_name)
display(analyze_button)

## 🎯 Interactive Location Selector
Enter your location coordinates below to get real-time weather analysis and disaster risk assessment.

In [ ]:
# 🔌 Open Meteo API Functions

def fetch_current_weather(latitude, longitude):
    """Fetch current weather data from Open Meteo API"""
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m,wind_direction_10m"
        response = requests.get(url, timeout=5)
        return response.json()
    except Exception as e:
        print(f"Error fetching current weather: {e}")
        return None

def fetch_forecast_weather(latitude, longitude, days=7):
    """Fetch weather forecast data"""
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,weather_code,wind_speed_10m_max&timezone=auto"
        response = requests.get(url, timeout=5)
        return response.json()
    except Exception as e:
        print(f"Error fetching forecast: {e}")
        return None

def get_risk_level(temp, humidity, wind_speed, precipitation):
    """Calculate disaster risk level based on weather parameters"""
    risk_score = 0
    factors = {}
    
    # Temperature risk
    if temp < -10 or temp > 50:
        risk_score += 30
        factors['temperature'] = 'Critical'
    elif temp < 0 or temp > 40:
        risk_score += 20
        factors['temperature'] = 'High'
    else:
        factors['temperature'] = 'Normal'
    
    # Humidity risk
    if humidity > 85:
        risk_score += 25
        factors['humidity'] = 'Critical'
    elif humidity > 70:
        risk_score += 15
        factors['humidity'] = 'High'
    else:
        factors['humidity'] = 'Normal'
    
    # Wind speed risk
    if wind_speed > 50:
        risk_score += 35
        factors['wind'] = 'Critical'
    elif wind_speed > 30:
        risk_score += 25
        factors['wind'] = 'High'
    elif wind_speed > 15:
        risk_score += 10
        factors['wind'] = 'Moderate'
    else:
        factors['wind'] = 'Low'
    
    # Precipitation risk
    if precipitation > 50:
        risk_score += 30
        factors['precipitation'] = 'Critical'
    elif precipitation > 20:
        risk_score += 20
        factors['precipitation'] = 'High'
    else:
        factors['precipitation'] = 'Low'
    
    risk_score = min(risk_score, 100)
    
    if risk_score >= 70:
        risk_level = "🔴 HIGH RISK"
    elif risk_score >= 40:
        risk_level = "🟡 MODERATE RISK"
    else:
        risk_level = "🟢 LOW RISK"
    
    return risk_score, risk_level, factors

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

# Set styling
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 8)

# 🌍 Interactive Disaster Risk Analysis with Open Meteo API
This notebook provides **real-time, dynamic analysis** of weather patterns and disaster risks using live data from the Open Meteo API.